# Network Intrusion Detection — NNConv Link Predictor
Full pipeline: data loading → subsampling → edge split → negative sampling → PyG Data → NNConv model → evaluation

## 1. Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import networkx as nx
import xml.etree.ElementTree as ET
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import NNConv

from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Helper functions — parse GraphML

In [ ]:
def parse_graphml(file_path):
    """Parse GraphML file and return NetworkX graph."""
    try:
        G = nx.read_graphml(file_path)
        return G
    except Exception:
        return parse_graphml_manual(file_path)


def parse_graphml_manual(file_path):
    """Manual fallback parsing of GraphML."""
    tree = ET.parse(file_path)
    root = tree.getroot()
    G = nx.Graph()

    keys = {}
    for key_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}key'):
        keys[key_elem.get('id')] = key_elem.get('attr.name')

    for node_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}node'):
        G.add_node(node_elem.get('id'))

    for edge_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}edge'):
        source = edge_elem.get('source')
        target = edge_elem.get('target')
        edge_attrs = {}
        for data_elem in edge_elem.findall('.//{http://graphml.graphdrawing.org/xmlns}data'):
            key_id = data_elem.get('key')
            attr_name = keys.get(key_id)
            if attr_name:
                try:
                    edge_attrs[attr_name] = float(data_elem.text)
                except Exception:
                    edge_attrs[attr_name] = data_elem.text
        G.add_edge(source, target, **edge_attrs)

    return G


def extract_edge_features(G):
    """Extract edge features as a pandas DataFrame."""
    rows = []
    for u, v, data in G.edges(data=True):
        row = {'source': u, 'target': v}
        row.update(data)
        rows.append(row)
    return pd.DataFrame(rows)

## 3. Load graph

In [ ]:
FILE_PATH = "/Users/dannykim/Desktop/vscode/Comp 395/Network-Intrusion-GNN/archive/0.1M-Stratified-Multi.graphml"

G_full = parse_graphml(FILE_PATH)
print(f"Full graph  — nodes: {G_full.number_of_nodes():,}  edges: {G_full.number_of_edges():,}")

# Extract edge features from the full graph for later use
edge_df = extract_edge_features(G_full)
print("\nEdge feature columns:", edge_df.columns.tolist())
print(edge_df.describe())

## 4. Subsample to a manageable size

The full graph has 41k nodes. Using a one-hot or degree feature matrix on that is expensive,
so we subsample to ~2000 nodes / 8000 edges for development.

In [ ]:
# random.seed(SEED)
# edges_full = list(G_full.edges(data=True))
# random.shuffle(edges_full)

# G_sub = nx.Graph()
# for u, v, data in edges_full:
#     G_sub.add_edge(u, v, **data)
#     if G_sub.number_of_nodes() >= 2000 and G_sub.number_of_edges() >= 8000:
#         break

# # Keep only the largest connected component
# largest_cc = max(nx.connected_components(G_sub), key=len)
# G_sub = G_sub.subgraph(largest_cc).copy()
# print(f"Subgraph    — nodes: {G_sub.number_of_nodes():,}  edges: {G_sub.number_of_edges():,}")

# # Relabel nodes to consecutive integers 0..N-1
# # (required by PyG — node indices must be integers)
# G_int = nx.convert_node_labels_to_integers(G_sub)
# print(f"Relabelled  — nodes: {G_int.number_of_nodes():,}  edges: {G_int.number_of_edges():,}")

# Sample nodes first, then take the induced subgraph
# This gives a more representative degree distribution
random.seed(SEED)
all_nodes = list(G_full.nodes())
sampled_nodes = random.sample(all_nodes, 3000)
G_sub = G_full.subgraph(sampled_nodes).copy()

# Keep largest connected component
largest_cc = max(nx.connected_components(G_sub), key=len)
G_sub = G_sub.subgraph(largest_cc).copy()
print(f"Subgraph — nodes: {G_sub.number_of_nodes():,}  edges: {G_sub.number_of_edges():,}")

# Relabel nodes to consecutive integers 0..N-1
# (required by PyG — node indices must be integers)
G_int = nx.convert_node_labels_to_integers(G_sub)
print(f"Relabelled  — nodes: {G_int.number_of_nodes():,}  edges: {G_int.number_of_edges():,}")

## 5. Edge train / val / test split

In [ ]:
all_edges = list(G_int.edges())
random.shuffle(all_edges)
n = len(all_edges)

train_edges = all_edges[:int(0.70 * n)]   # 70%
val_edges   = all_edges[int(0.70 * n):int(0.85 * n)]  # 15%
test_edges  = all_edges[int(0.85 * n):]   # 15%

print(f"Train edges: {len(train_edges):,}")
print(f"Val edges:   {len(val_edges):,}")
print(f"Test edges:  {len(test_edges):,}")

# Training graph — only contains train edges to prevent data leakage
G_train = nx.Graph()
G_train.add_nodes_from(G_int.nodes(data=True))
G_train.add_edges_from(
    (u, v, G_int[u][v]) for u, v in train_edges
)

## 6. Negative edge sampling

For each split we generate an equal number of fake (non-existent) edges.
These act as the negative class for the binary classification problem.

In [ ]:
def generate_negative_edges(G, num_samples, seed=42):
    """Sample edges that do NOT exist in G."""
    random.seed(seed)
    neg = []
    nodes = list(G.nodes())
    existing = set(G.edges()) | {(v, u) for u, v in G.edges()}
    while len(neg) < num_samples:
        u, v = random.sample(nodes, 2)
        if (u, v) not in existing:
            neg.append((u, v))
    return neg


neg_train = generate_negative_edges(G_train, len(train_edges), seed=42)
neg_val   = generate_negative_edges(G_train, len(val_edges),   seed=43)
neg_test  = generate_negative_edges(G_train, len(test_edges),  seed=44)

print(f"Negative train: {len(neg_train):,}")
print(f"Negative val:   {len(neg_val):,}")
print(f"Negative test:  {len(neg_test):,}")

## 7. Build node index map and integer edge lists

`node2idx` maps each node ID to a row index in the feature matrix.
All edge lists must use these integer indices for PyG.

In [ ]:
# node2idx: node_id -> integer index
node2idx = {n: i for i, n in enumerate(G_train.nodes())}


def to_idx(edges):
    """Convert a list of (u, v) node-ID pairs to integer index pairs.
    Drops any edge whose endpoints are not in node2idx (e.g. isolated val/test nodes).
    """
    return [
        (node2idx[u], node2idx[v])
        for u, v in edges
        if u in node2idx and v in node2idx
    ]


train_pos_idx = to_idx(train_edges)
val_pos_idx   = to_idx(val_edges)
test_pos_idx  = to_idx(test_edges)
neg_train_idx = to_idx(neg_train)
neg_val_idx   = to_idx(neg_val)
neg_test_idx  = to_idx(neg_test)

print(f"train_pos_idx: {len(train_pos_idx):,}  neg_train_idx: {len(neg_train_idx):,}")
print(f"val_pos_idx:   {len(val_pos_idx):,}  neg_val_idx:   {len(neg_val_idx):,}")
print(f"test_pos_idx:  {len(test_pos_idx):,}  neg_test_idx:  {len(neg_test_idx):,}")

## 8. Build PyG Data object with edge features

Node features: normalised degree (1 scalar per node — cheap and informative).
Edge features: the 7 traffic columns from the dataset.

In [ ]:
# EDGE_FEAT_COLS = ['TotPkts', 'TotBytes', 'SrcBytes', 'Dur',
#                   'Proto_encoded', 'Dir_encoded', 'State_encoded']
# NUM_EDGE_FEATS = len(EDGE_FEAT_COLS)  # 7
NUM_NODE_FEATS = 1                     # degree

# # ── Node features: normalised degree ──────────────────────────────────────────
# degrees = np.array([G_train.degree(n) for n in G_train.nodes()], dtype=float)
# degrees = (degrees - degrees.mean()) / (degrees.std() + 1e-9)
# x = torch.tensor(degrees, dtype=torch.float).unsqueeze(1)  # shape [N, 1]

# # ── Edge feature lookup: (u, v) -> feature vector ─────────────────────────────
# # We build this from edge_df (extracted from the full graph earlier).
# # For edges missing from edge_df we fall back to zeros.
# edge_feat_dict = {}
# zero_feats = np.zeros(NUM_EDGE_FEATS)

# for _, row in edge_df.iterrows():
#     u_raw, v_raw = row['source'], row['target']
#     # edge_df uses original string node IDs; we need integer IDs from G_int
#     # G_int was relabelled from G_sub, so we look up via the mapping stored
#     # inside G_sub's relabelling.  Simplest approach: check both directions.
#     feats = []
#     for col in EDGE_FEAT_COLS:
#         val = row.get(col, 0.0)
#         feats.append(float(val) if pd.notna(val) else 0.0)
#     feats = np.array(feats)
#     edge_feat_dict[(u_raw, v_raw)] = feats
#     edge_feat_dict[(v_raw, u_raw)] = feats

# # ── Edge index and edge_attr for the training graph ───────────────────────────
# train_edge_list = list(G_train.edges())

# edge_index = torch.tensor(
#     [[node2idx[u], node2idx[v]] for u, v in train_edge_list],
#     dtype=torch.long
# ).t().contiguous()  # shape [2, E]

# edge_attr = []
# for u, v in train_edge_list:
#     # G_train stores integer node IDs; edge_feat_dict has original string IDs.
#     # G_int carries the original label in node attribute 'label' if available,
#     # but simplest fallback: use the raw edge data stored on G_train itself.
#     raw_data = G_train[u][v]
#     feats = []
#     for col in EDGE_FEAT_COLS:
#         val = raw_data.get(col, 0.0)
#         feats.append(float(val) if val is not None else 0.0)
#     edge_attr.append(feats)

# edge_attr = torch.tensor(edge_attr, dtype=torch.float)  # shape [E, 7]

# data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

# print("Data object:")
# print(f"  x:          {data.x.shape}")
# print(f"  edge_index: {data.edge_index.shape}")
# print(f"  edge_attr:  {data.edge_attr.shape}")

# Aggregate edge features per node as node features
# For each node, compute mean of its edges' traffic values
EDGE_FEAT_COLS = ['TotPkts', 'TotBytes', 'SrcBytes', 'Dur',
                  'Proto_encoded', 'Dir_encoded', 'State_encoded']


NUM_EDGE_FEATS = len(EDGE_FEAT_COLS)  # 7
NUM_NODE_FEATS = 1                     # degree


node_feat_dict = {n: [] for n in G_train.nodes()}
for u, v, edata in G_train.edges(data=True):
    feats = [float(edata.get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
    node_feat_dict[u].append(feats)
    node_feat_dict[v].append(feats)

node_feats = []
for n in G_train.nodes():
    edges_feats = node_feat_dict[n]
    if edges_feats:
        node_feats.append(np.mean(edges_feats, axis=0))
    else:
        node_feats.append(np.zeros(len(EDGE_FEAT_COLS)))

node_feats = np.array(node_feats)

# Normalize
node_feats = (node_feats - node_feats.mean(axis=0)) / (node_feats.std(axis=0) + 1e-9)
x = torch.tensor(node_feats, dtype=torch.float)  # shape [N, 7]


# ── Edge feature lookup: (u, v) -> feature vector ─────────────────────────────
# We build this from edge_df (extracted from the full graph earlier).
# For edges missing from edge_df we fall back to zeros.
edge_feat_dict = {}
zero_feats = np.zeros(NUM_EDGE_FEATS)

for _, row in edge_df.iterrows():
    u_raw, v_raw = row['source'], row['target']
    # edge_df uses original string node IDs; we need integer IDs from G_int
    # G_int was relabelled from G_sub, so we look up via the mapping stored
    # inside G_sub's relabelling.  Simplest approach: check both directions.
    feats = []
    for col in EDGE_FEAT_COLS:
        val = row.get(col, 0.0)
        feats.append(float(val) if pd.notna(val) else 0.0)
    feats = np.array(feats)
    edge_feat_dict[(u_raw, v_raw)] = feats
    edge_feat_dict[(v_raw, u_raw)] = feats

# ── Edge index and edge_attr for the training graph ───────────────────────────
train_edge_list = list(G_train.edges())

edge_index = torch.tensor(
    [[node2idx[u], node2idx[v]] for u, v in train_edge_list],
    dtype=torch.long
).t().contiguous()  # shape [2, E]

edge_attr = []
for u, v in train_edge_list:
    # G_train stores integer node IDs; edge_feat_dict has original string IDs.
    # G_int carries the original label in node attribute 'label' if available,
    # but simplest fallback: use the raw edge data stored on G_train itself.
    raw_data = G_train[u][v]
    feats = []
    for col in EDGE_FEAT_COLS:
        val = raw_data.get(col, 0.0)
        feats.append(float(val) if val is not None else 0.0)
    edge_attr.append(feats)

edge_attr = torch.tensor(edge_attr, dtype=torch.float)  # shape [E, 7]

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

print("Data object:")
print(f"  x:          {data.x.shape}")
print(f"  edge_index: {data.edge_index.shape}")
print(f"  edge_attr:  {data.edge_attr.shape}")


## 9. NNConv model

NNConv uses edge features to compute a per-edge weight matrix that transforms
the source node's features before aggregating into the target node.
This means every edge contributes differently to the message passing,
which is exactly what we want when traffic features carry the signal.

In [ ]:
HIDDEN = 64


class GNN(nn.Module):
    def __init__(self, node_in, edge_in, hidden):
        super().__init__()

        # Conv 1: node_in -> hidden
        # edge_nn output must equal node_in * hidden (produces the weight matrix)
        self.conv1 = NNConv(
            in_channels=node_in,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, node_in * hidden),
                nn.ReLU()
            )
        )

        # Conv 2: hidden -> hidden
        self.conv2 = NNConv(
            in_channels=hidden,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, hidden * hidden),
                nn.ReLU()
            )
        )

        # MLP decoder: concat of two node embeddings -> edge score
        self.mlp = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )

    def encode(self, x, edge_index, edge_attr):
        """Run message passing to produce node embeddings."""
        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = self.conv2(x, edge_index, edge_attr)
        return x

    def decode(self, z, edges):
        """Score candidate edges from node embeddings."""
        u = torch.tensor([e[0] for e in edges], dtype=torch.long)
        v = torch.tensor([e[1] for e in edges], dtype=torch.long)
        return self.mlp(torch.cat([z[u], z[v]], dim=1)).squeeze()

    def forward(self, x, edge_index, edge_attr, edges):
        z = self.encode(x, edge_index, edge_attr)
        return self.decode(z, edges)


model = GNN(
    node_in=NUM_NODE_FEATS,
    edge_in=NUM_EDGE_FEATS,
    hidden=HIDDEN
)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

## 10. Training

In [ ]:
def evaluate(model, data, pos_edges, neg_edges, mask_edge_index=None):
    model.eval()
    with torch.no_grad():
        edge_index = mask_edge_index if mask_edge_index is not None else data.edge_index
        scores = model(
            data.x, edge_index, data.edge_attr,
            pos_edges + neg_edges
        )
    y = [1] * len(pos_edges) + [0] * len(neg_edges)
    s = scores.numpy()
    return roc_auc_score(y, s), average_precision_score(y, s)


optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
EPOCHS = 50

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    all_train_edges = train_pos_idx + neg_train_idx
    preds = model(data.x, data.edge_index, data.edge_attr, all_train_edges)
    labels = torch.tensor(
        [1.0] * len(train_pos_idx) + [0.0] * len(neg_train_idx)
    )

    loss = F.binary_cross_entropy(preds, labels)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        auc, ap = evaluate(model, data, val_pos_idx, neg_val_idx)
        print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f} "
              f"| Val AUC: {auc:.4f} | Val AP: {ap:.4f}")

## 11. Final test evaluation

In [ ]:
test_auc, test_ap = evaluate(model, data, test_pos_idx, neg_test_idx)
print(f"Test AUC: {test_auc:.4f}")
print(f"Test AP:  {test_ap:.4f}")

In [ ]:
#sanity check

# Shuffle the test labels
import copy
y_true = [1] * len(test_pos_idx) + [0] * len(neg_test_idx)
y_shuffled = copy.copy(y_true)
random.shuffle(y_shuffled)

model.eval()
with torch.no_grad():
    scores = model(data.x, data.edge_index, data.edge_attr,
                   test_pos_idx + neg_test_idx).numpy()

print("Real AUC:    ", roc_auc_score(y_true, scores))
print("Shuffled AUC:", roc_auc_score(y_shuffled, scores))

# Also check: what does a degree-only baseline get?
deg = dict(G_train.degree())
deg_scores = [deg.get(u, 0) + deg.get(v, 0) for u, v in test_pos_idx + neg_test_idx]
print("Degree baseline AUC:", roc_auc_score(y_true, deg_scores))